In [ ]:
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os
import tensorflow as tf, tensorflow.keras as keras, tensorflow_datasets as tfds
from keras.layers import Flatten, Dense, Dropout, MaxPooling2D, Conv2D, GlobalAveragePooling2D
from keras.models import Sequential
from tensorflow.keras.applications import resnet50, xception, mobilenet, mobilenet_v2, mobilenet_v3
from tensorflow.keras.utils import image_dataset_from_directory as idfd
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals
tDIR, sDIR = 'train/', 'test/'

In [ ]:
from sklearn.model_selection import StratifiedKFold,train_test_split
from sklearn.metrics import accuracy_score
from pathlib import Path

In [ ]:
def get_paths_labels(data_dir):
    # Get sorted image paths
    paths = sorted([str(p) for p in Path(data_dir).rglob("*")
                    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
                    ])
    # Map class names to integer labels
    class_names = sorted([p.name for p in Path(data_dir).iterdir() if p.is_dir()])

    class_to_idx = {name: i for i, name in enumerate(class_names)}

    labels = np.array([class_to_idx[Path(path).parent.name]
                        for path in paths
                        ])
    paths = np.array(paths)
    return paths, labels

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(paths, labels, batch_size, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            len(paths),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [ ]:
def build_wide_model(tensor_shape=(218, 178, 3), num_classes=2, BS=None, EPO=None, TS=None, pretrain_base=None):
    tf.keras.utils.set_random_seed(0)
    Init = keras.initializers.RandomNormal(seed=0)

    # Shared Input
    input_layer = keras.layers.Input(shape=tensor_shape, name="input_layer")
    resize_input = tf.keras.layers.Resizing(
        height=224,
        width=224,
        crop_to_aspect_ratio=True
    )(input_layer)

    # Pathway 1: Pretrained Feature Extractor (MobileNetV2)
    x_pre = pretrain_base(resize_input)
    x_pre = keras.layers.GlobalAveragePooling2D()(x_pre)
    #---------------------------------------------------------------------------
    # Pathway 2: Custom CNN Pathway
    h = Conv2D(filters=32, kernel_size=(5, 5), padding="same", strides=3,
                kernel_initializer=Init)(resize_input)
    # h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)
    # h = MaxPooling2D(2)(h)
    # h = keras.layers.SpatialDropout2D(0.1)(h)

    h = Conv2D(filters=64, kernel_size=(3, 3), padding="same", strides=2,
                kernel_initializer=Init)(h)
    # h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)
    # h = MaxPooling2D(2)(h)
    # h = keras.layers.SpatialDropout2D(0.1)(h)

    h = Conv2D(filters=128, kernel_size=(3, 3), padding="same", strides=1,
                kernel_initializer=Init)(h)
    h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)
    h = MaxPooling2D(2)(h)
    # h = keras.layers.SpatialDropout2D(0.3)(h)

    h = Conv2D(filters=256, kernel_size=(2, 2), padding="same", strides=1,
                kernel_initializer=Init)(h)
    h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)
    h = MaxPooling2D(2)(h)
    h = keras.layers.SpatialDropout2D(0.1)(h)

    h = Flatten()(h)

    #---------------------------------------------------------------------------
    # Wide Concatenation of Feature Pathways
    concatenated = keras.layers.concatenate([x_pre,h])

    # Final Classification Dense block
    dense = Dense(64, kernel_initializer=Init)(concatenated)
    dense = keras.layers.BatchNormalization()(dense)
    dense = keras.layers.Activation("relu")(dense)
    # dense = Dropout(rate=0.2)(dense)

    output_layer = keras.layers.Dense(1, activation="sigmoid")(dense)

    m = keras.Model(inputs=input_layer, outputs=output_layer)

    m.compile(
        loss="binary_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
        metrics=["accuracy"]
    )
    return m

In [ ]:
def build_wide_model_stable(tensor_shape=(218, 178, 3), num_classes=2):
    """
    Instantiates the pretrained MobileNetV2 base inside the builder
    to guarantee fold isolation and weight stability during CV.
    """
    tf.keras.utils.set_random_seed(0)
    Init = keras.initializers.RandomNormal(seed=0)

    # Instantiate a clean pretrained base inside the function
    pretrain_base = mobilenet_v2.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )
    # pretrain_base = ConvNeXtTiny(
    #     input_shape=(224, 224, 3),
    #     include_top=False,
    #     weights="imagenet")
    pretrain_base.trainable = False

    # Shared Input
    input_layer = keras.layers.Input(shape=tensor_shape, name="input_layer")
    resize_input = tf.keras.layers.Resizing(
        height=224,
        width=224,
        crop_to_aspect_ratio=True
    )(input_layer)

    # Pathway 1: Isolated Pretrained Feature Extractor
    x_pre = pretrain_base(resize_input)
    x_pre = keras.layers.GlobalAveragePooling2D()(x_pre)
    # Pathway 2: Custom CNN Pathway
    h = Conv2D(filters=32, kernel_size=(5, 5), padding="same", strides=3, kernel_initializer=Init)(resize_input)
    # h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)

    h = Conv2D(filters=64, kernel_size=(3, 3), padding="same", strides=2, kernel_initializer=Init)(h)
    # h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)

    h = Conv2D(filters=128, kernel_size=(3, 3), padding="same", strides=1, kernel_initializer=Init)(h)
    h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)
    h = MaxPooling2D(2)(h)

    h = Conv2D(filters=256, kernel_size=(2, 2), padding="same", strides=1, kernel_initializer=Init)(h)
    h = keras.layers.BatchNormalization()(h)
    h = keras.layers.Activation("relu")(h)
    h = MaxPooling2D(2)(h)
    h = keras.layers.SpatialDropout2D(0.25)(h)

    h = Flatten()(h)

    # Wide Concatenation of Feature Pathways
    concatenated = keras.layers.concatenate([x_pre, h])
    # Final Classification Dense block
    concatenated = Dense(64, kernel_initializer=Init)(concatenated)
    concatenated = keras.layers.BatchNormalization()(concatenated)
    concatenated = keras.layers.Activation("relu")(concatenated)
    output_layer = keras.layers.Dense(1, activation="sigmoid")(concatenated)

    m = keras.Model(inputs=input_layer, outputs=output_layer)
    m.compile(
        loss="binary_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
        metrics=["accuracy"]
    )
    return m

In [ ]:
# --- Stratified CV Loop with Horizontally Flipped Validation Dataset ---
N_FOLDS = 4
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
accu = []
accu2 = []
accu3 = []

img_path, labels = get_paths_labels(tDIR)
print("=== 4-Fold Stratified CV with Horizontally Flipped Validation Dataset ===")
for fold, (train_idx, val_idx) in enumerate(cv.split(img_path, labels)):
    print(f"\nFold {fold + 1}/{N_FOLDS}, train size: {len(train_idx)}")
    BS = 192
    EPO = 30
    train_ds = make_dataset(img_path[train_idx], labels[train_idx], batch_size=BS, training=True)

    # Generate standard validation dataset
    val_ds = make_dataset(img_path[val_idx], labels[val_idx], batch_size=BS)

    # Map tf.image.flip_left_right to flip all validation images horizontally
    val_ds_flipped = val_ds.map(
        lambda x, y: (tf.image.flip_left_right(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    model = build_wide_model_stable()
    model.fit(train_ds, epochs=EPO, batch_size=BS, verbose=0)
    prob_1 = model.predict(val_ds)
    pred_1 = (prob_1 > 0.5).astype(int)
    prob_2 = model.predict(val_ds_flipped)
    pred_2 = (prob_2 > 0.5).astype(int)
    prob_avg = (prob_1 + prob_2) / 2
    pred_avg = (prob_avg > 0.5).astype(int)
    compare = (pred_1 == pred_2).astype(int)
    print(np.unique(compare, return_counts=True))
    accu_score = accuracy_score(labels[val_idx], pred_1)
    accu_score2 = accuracy_score(labels[val_idx], pred_2)
    accu_score3 = accuracy_score(labels[val_idx], pred_avg)
    print(f"Fold {fold + 1} : accu = {accu_score:.6f}")
    print(f"Fold {fold + 1} : accu = {accu_score2:.6f}")
    print(f"Fold {fold + 1} : accu = {accu_score3:.6f}")
    # results = model.evaluate(val_ds, return_dict=True)
    # results2 = model.evaluate(val_ds_flipped, return_dict=True)

    accu += [accu_score]
    accu2 += [accu_score2]
    accu3 += [accu_score3]

print(f"Mean accu: {np.mean(accu):.6f}|{np.std(accu):.6f}")
print(f"Mean accu: {np.mean(accu2):.6f}|{np.std(accu2):.6f}")
print(f"Mean accu: {np.mean(accu3):.6f}|{np.std(accu3):.6f}")

|mean accu|std|model|
|-|-|-|
|0.944000|0.002530|cv(32,5,3)(64,3,2)(128,3,1)(256,2,1),30epoc,192BS,lr=0.002,bnorm(cv3,4),maxp(3,4),drp(4,0.25),concat MNV2,Dense(64,BN),orig and flipped inference avg|

In [ ]:
BS = 192
EPO = 30

# Prepare the full train dataset using your custom generator
img_path, labels = get_paths_labels(tDIR)
full_train_ds = make_dataset(img_path, labels, batch_size=BS, training=True)

# Re-build and train the wide model on the full dataset using the stable builder
final_model = build_wide_model_stable()
final_model.fit(full_train_ds, epochs=EPO, batch_size=BS, verbose=0)

# Construct the test dataset using the exact same load_image preprocessing pipeline
test_paths = sorted([str(p) for p in Path(sDIR).rglob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
dummy_labels = np.zeros(len(test_paths))  # dummy labels needed for the generator map function

custom_test_ds = tf.data.Dataset.from_tensor_slices((test_paths, dummy_labels))
custom_test_ds = custom_test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
custom_test_ds = custom_test_ds.batch(BS)
custom_test_ds = custom_test_ds.prefetch(tf.data.AUTOTUNE)

# Construct the horizontally flipped version of the test dataset for TTA
custom_test_ds_flipped = custom_test_ds.map(
    lambda x, y: (tf.image.flip_left_right(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Generate predictions for both versions of the test dataset
prob_orig = final_model.predict(custom_test_ds)
prob_flipped = final_model.predict(custom_test_ds_flipped)

# Average predictions (TTA: Test-Time Augmentation)
y_pred = (prob_orig + prob_flipped) / 2.0

# Extract file IDs in matching sorted order
vIDs = [Path(p).stem for p in test_paths]

In [ ]:
# Map the predicted probabilities to binary outputs using the optimal standard 0.5 threshold as float values (1.0 / 0.0)
pY = pd.DataFrame((y_pred > 0.5).astype(int), index=vIDs, columns=['y'])
ToCSV(pY, 'submission-final')

In [ ]:
pY.value_counts()